In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from broflow import (
    BaseTask, TaskRegistry, Flow, 
    to_edges, to_tree, to_mermaid
)

In [3]:
from dataclasses import dataclass
from enum import StrEnum
import random

class Process(StrEnum):
    START = 'start'
    INPUT = 'input'
    ROUTER = 'router'
    TOOL_SELECTION = 'tool_selection'
    TOOL_EXECUTION = 'tool_execution'
    ANSWER = 'answer'
    FINISH = 'finish'

@dataclass
class State:
    input:str = ''
    tool_use:str = ''
    tool_result:str = ''
    answer:str = ''

class UserInput(BaseTask):
    possible_next = {Process.ROUTER}
    def __call__(self, state:State):
        state.input = "Test input"
        self.set_next(Process.ROUTER)
        print(f"{self.__class__} -> {self.next}")
        return state

class Router(BaseTask):
    possible_next = {Process.ANSWER, Process.TOOL_SELECTION}
    def __call__(self, state:State):
        self.set_next(random.choice([Process.ANSWER, Process.TOOL_SELECTION]))
        print(f"{self.__class__} -> {self.next}")
        return state

class ToolSelection(BaseTask):
    possible_next = {Process.TOOL_EXECUTION}
    def __call__(self, state:State):
        state.tool_use = "some tool"
        self.set_next(Process.TOOL_EXECUTION)
        print(f"{self.__class__} -> {self.next}")
        return state

class ToolExecution(BaseTask):
    possible_next = {Process.ANSWER}
    def __call__(self, state:State):
        state.tool_result = "some result"
        self.set_next(Process.ANSWER)
        print(f"{self.__class__} -> {self.next}")
        return state

class Answer(BaseTask):
    possible_next = {Process.INPUT, Process.FINISH}
    def __call__(self, state:State):
        self.set_next(random.choice([Process.INPUT, Process.FINISH]))
        state.answer = f"Answer is: {state.tool_result}"
        print(f"{self.__class__} -> {self.next}")
        return state

In [4]:
registry = TaskRegistry()
registry.register(Process.INPUT, UserInput(name="input"))
registry.register(Process.ROUTER, Router(name="router"))
registry.register(Process.TOOL_SELECTION, ToolSelection(name="tool_selection"))
registry.register(Process.TOOL_EXECUTION, ToolExecution(name="tool_execution"))
registry.register(Process.ANSWER, Answer(name="answer"))

In [5]:

print(to_tree(registry, start=Process.INPUT, terminate=Process.FINISH))

input
  -> router
    -> answer
      -> finish
      -> input
    -> tool_selection
      -> tool_execution
        -> answer


In [6]:
print(to_mermaid(registry, start=Process.INPUT, terminate=Process.FINISH))

```mermaid
flowchart TD
    input --> router
    router -.-> answer
    answer -.-> finish
    answer -.-> input
    router -.-> tool_selection
    tool_selection --> tool_execution
    tool_execution --> answer
```


In [7]:
print(to_edges(registry))

[('answer', 'finish'), ('answer', 'input'), ('input', 'router'), ('router', 'answer'), ('router', 'tool_selection'), ('tool_execution', 'answer'), ('tool_selection', 'tool_execution')]


In [8]:
state = State()
flow = Flow(registry)
final_state = flow.run(start=Process.INPUT, end=Process.FINISH, state=state)

<class '__main__.UserInput'> -> router
<class '__main__.Router'> -> tool_selection
<class '__main__.ToolSelection'> -> tool_execution
<class '__main__.ToolExecution'> -> answer
<class '__main__.Answer'> -> input
<class '__main__.UserInput'> -> router
<class '__main__.Router'> -> answer
<class '__main__.Answer'> -> input
<class '__main__.UserInput'> -> router
<class '__main__.Router'> -> tool_selection
<class '__main__.ToolSelection'> -> tool_execution
<class '__main__.ToolExecution'> -> answer
<class '__main__.Answer'> -> finish


In [9]:
final_state

State(input='Test input', tool_use='some tool', tool_result='some result', answer='Answer is: some result')

In [10]:
flow.trace

[('input', <Process.ROUTER: 'router'>),
 ('router', <Process.TOOL_SELECTION: 'tool_selection'>),
 ('tool_selection', <Process.TOOL_EXECUTION: 'tool_execution'>),
 ('tool_execution', <Process.ANSWER: 'answer'>),
 ('answer', <Process.INPUT: 'input'>),
 ('input', <Process.ROUTER: 'router'>),
 ('router', <Process.ANSWER: 'answer'>),
 ('answer', <Process.INPUT: 'input'>),
 ('input', <Process.ROUTER: 'router'>),
 ('router', <Process.TOOL_SELECTION: 'tool_selection'>),
 ('tool_selection', <Process.TOOL_EXECUTION: 'tool_execution'>),
 ('tool_execution', <Process.ANSWER: 'answer'>),
 ('answer', <Process.FINISH: 'finish'>)]